<a href="https://colab.research.google.com/github/Shineii86/GalaxyBrain/blob/main/notebooks/GalaxyBrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">
  <img src="https://raw.githubusercontent.com/Shineii86/GalaxyBrain/main/images/GalaxyBrain.png" width="300px" alt="Galaxy Brain Achievement">
  <h1>🧠 Galaxy Brain Achievement Helper</h1>
  <p><b>Automate GitHub Discussions answers to unlock the Galaxy Brain badge — all from your browser.</b></p>
</div>

---

> ⚠️ **IMPORTANT**
> - This script uses the GitHub GraphQL API to create discussions and mark answers as accepted.
> - **Galaxy Brain requires at least two accepted answers in GitHub Discussions.** This tool automates the process using secondary accounts.
> - **Use responsibly.** Artificially inflating achievements may be viewed negatively by potential employers or collaborators, and may violate GitHub's Terms of Service.
> - You need **Personal Access Tokens (classic)** with `repo` and `write:discussion` scopes for each account.
> - The script will create discussions and answers in the specified repository. Ensure the repository has **Discussions enabled** and a **Q&A category**.

---

In [ ]:
#@title 📦 1. Install Dependencies & Setup
!pip install -q requests

import os
import json
import time
import random
import requests
from datetime import datetime

print("✅ All dependencies installed and imported.")

In [ ]:
#@title ⚙️ 2. Configuration & Run

# =============================================================================
#  🔧 Configuration
# =============================================================================

# Your main GitHub username (the one that will earn the achievement)
MAIN_USERNAME = "Shineii86"  #@param {type:"string"}

# Main account Personal Access Token (classic) with 'repo' and 'write:discussion' scopes
MAIN_TOKEN = "ghp_xxxxxxxxxxxxxxxxxxxx"  #@param {type:"string"}

# Secondary account #1 username
USERNAME_1 = "secondary1"  #@param {type:"string"}

# Secondary account #1 Personal Access Token (classic) with 'repo' and 'write:discussion' scopes
TOKEN_1 = "ghp_xxxxxxxxxxxxxxxxxxxx"  #@param {type:"string"}

# Secondary account #2 username
USERNAME_2 = "secondary2"  #@param {type:"string"}

# Secondary account #2 Personal Access Token (classic) with 'repo' and 'write:discussion' scopes
TOKEN_2 = "ghp_xxxxxxxxxxxxxxxxxxxx"  #@param {type:"string"}

# Repository name (must exist under MAIN_USERNAME with Discussions enabled)
REPO_NAME = "galaxy-brain-demo"  #@param {type:"string"}

# Number of accepted answers to create (minimum 2 for Galaxy Brain)
NUM_ANSWERS = 2  #@param {type:"integer"}

# Delay between actions (seconds) – increase to appear more natural
ACTION_DELAY = 5  #@param {type:"number"}

# =============================================================================
#  🧠 Galaxy Brain Automation Script
# =============================================================================

GITHUB_GRAPHQL_URL = "https://api.github.com/graphql"

def graphql_request(token, query, variables=None):
    """Execute a GraphQL request and return the JSON response."""
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    payload = {"query": query}
    if variables:
        payload["variables"] = variables
    
    response = requests.post(GITHUB_GRAPHQL_URL, json=payload, headers=headers)
    if response.status_code != 200:
        raise Exception(f"GraphQL request failed: {response.status_code} - {response.text}")
    
    data = response.json()
    if "errors" in data:
        raise Exception(f"GraphQL errors: {data['errors']}")
    return data

def get_repository_id_and_answerable_category(token, owner, repo_name):
    """Fetch repository ID and the first answerable discussion category."""
    query = """
    query($owner: String!, $name: String!) {
        repository(owner: $owner, name: $name) {
            id
            discussionCategories(first: 20) {
                nodes {
                    id
                    name
                    isAnswerable
                }
            }
        }
    }
    """
    data = graphql_request(token, query, {"owner": owner, "name": repo_name})
    repo = data["data"]["repository"]
    if not repo:
        raise Exception(f"Repository {owner}/{repo_name} not found or no access.")
    
    categories = repo["discussionCategories"]["nodes"]
    if not categories:
        raise Exception("Repository has no discussion categories. Enable Discussions first.")
    
    # Find first answerable category
    answerable_category = next((cat for cat in categories if cat["isAnswerable"]), None)
    
    if not answerable_category:
        print("\n❌ No answerable category found!")
        print("Please create a 'Q&A' category in your repository:")
        print(f"  1. Go to https://github.com/{owner}/{repo_name}/discussions/categories")
        print("  2. Click 'New category'")
        print("  3. Choose 'Q&A' format and give it a name (e.g., 'Q&A')")
        print("  4. Save and re-run this script.\n")
        raise Exception("No answerable discussion category available.")
    
    print(f"✅ Found repository ID and answerable category: '{answerable_category['name']}'")
    return repo["id"], answerable_category["id"]

def create_discussion(token, repo_id, category_id, title, body):
    """Create a new discussion and return its ID."""
    query = """
    mutation($repoId: ID!, $categoryId: ID!, $title: String!, $body: String!) {
        createDiscussion(input: {
            repositoryId: $repoId,
            categoryId: $categoryId,
            title: $title,
            body: $body
        }) {
            discussion {
                id
                url
            }
        }
    }
    """
    variables = {
        "repoId": repo_id,
        "categoryId": category_id,
        "title": title,
        "body": body
    }
    data = graphql_request(token, query, variables)
    discussion = data["data"]["createDiscussion"]["discussion"]
    print(f"   📝 Discussion created: {discussion['url']}")
    return discussion["id"]

def add_discussion_comment(token, discussion_id, body):
    """Add a comment (answer) to a discussion and return the comment ID."""
    query = """
    mutation($discussionId: ID!, $body: String!) {
        addDiscussionComment(input: {
            discussionId: $discussionId,
            body: $body
        }) {
            comment {
                id
            }
        }
    }
    """
    variables = {"discussionId": discussion_id, "body": body}
    data = graphql_request(token, query, variables)
    comment_id = data["data"]["addDiscussionComment"]["comment"]["id"]
    print(f"   💬 Answer posted by secondary account")
    return comment_id

def mark_comment_as_answer(token, comment_id):
    """Mark a discussion comment as the accepted answer."""
    query = """
    mutation($commentId: ID!) {
        markDiscussionCommentAsAnswer(input: {
            id: $commentId
        }) {
            discussion {
                id
            }
        }
    }
    """
    data = graphql_request(token, query, {"commentId": comment_id})
    print(f"   ✅ Answer marked as accepted")
    return True

# =============================================================================
#  🚀 Main Execution
# =============================================================================

print(f"\n🧠 Galaxy Brain Automation for user '{MAIN_USERNAME}'")
print(f"Repository: {MAIN_USERNAME}/{REPO_NAME}")
print(f"Creating {NUM_ANSWERS} accepted answer(s)\n")

# List of secondary accounts (you can add more if needed)
secondary_accounts = [
    {"username": USERNAME_1, "token": TOKEN_1},
    {"username": USERNAME_2, "token": TOKEN_2}
]

# Ensure we have enough secondary accounts
if NUM_ANSWERS > len(secondary_accounts):
    print(f"⚠️ Warning: {NUM_ANSWERS} answers requested but only {len(secondary_accounts)} secondary accounts provided.")
    print("   Answers will reuse secondary accounts if necessary.\n")

# Step 1: Get repository ID and an answerable category ID (using main account)
print("🔍 Fetching repository information...")
try:
    repo_id, category_id = get_repository_id_and_answerable_category(MAIN_TOKEN, MAIN_USERNAME, REPO_NAME)
except Exception as e:
    print(f"❌ Error: {e}")
    print("\nPlease ensure:")
    print("  - The repository exists and you have admin/write access.")
    print("  - Discussions are enabled (Settings → Features → Discussions).")
    print("  - At least one 'Q&A' category exists (Settings → Discussions → Categories).")
    print("  - Your token has 'repo' and 'write:discussion' scopes.")
    raise

time.sleep(ACTION_DELAY)

# Step 2: Create discussions and answers
accepted_count = 0

for i in range(NUM_ANSWERS):
    print(f"\n--- Processing answer {i+1} of {NUM_ANSWERS} ---")
    
    # Select a secondary account (round-robin)
    sec = secondary_accounts[i % len(secondary_accounts)]
    print(f"👤 Secondary account: {sec['username']}")
    
    # Create a unique discussion title
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    title = f"Question about feature X - {timestamp}"
    body = f"This is an automated test discussion to demonstrate the answer acceptance flow.\n\n**Question:** How do I implement feature X efficiently?"
    
    print(f"📝 Creating discussion...")
    discussion_id = create_discussion(MAIN_TOKEN, repo_id, category_id, title, body)
    time.sleep(ACTION_DELAY)
    
    # Secondary account posts an answer
    print(f"💬 Posting answer as {sec['username']}...")
    answer_body = f"Here's a detailed solution to your question:\n\n1. First, consider using approach A.\n2. Then, implement B for optimization.\n3. Finally, test with C.\n\nHope this helps!"
    comment_id = add_discussion_comment(sec['token'], discussion_id, answer_body)
    time.sleep(ACTION_DELAY)
    
    # Main account marks the answer as accepted
    print(f"✅ Marking answer as accepted...")
    mark_comment_as_answer(MAIN_TOKEN, comment_id)
    time.sleep(ACTION_DELAY)
    
    accepted_count += 1
    print(f"🎉 Answer {accepted_count} accepted!")

print("\n" + "="*50)
print(f"✨ Success! Created {accepted_count} accepted answer(s).")
print(f"📊 Check your profile: https://github.com/{MAIN_USERNAME}")
print("\n🔔 Note: Achievements may take up to 24 hours to appear.")
print("   You can also check: https://github.com/settings/achievements")
print("\n---")
print("🧠 Galaxy Brain Helper By [Shinei Nouzen](https://github.com/Shineii86/GalaxyBrain)")


---

### 📚 How It Works

- The script uses the **GitHub GraphQL API** to interact with Discussions.
- It first fetches the repository ID and an available **answerable** discussion category (Q&A format).
- For each desired answer:
  1. Creates a new discussion using the **main account**.
  2. Posts an answer (comment) using a **secondary account**.
  3. Marks that answer as **accepted** using the main account.
- Delays are added between actions to simulate natural behavior.

### ⚠️ Important Notes

- **Repository must have Discussions enabled** (Settings → Features → Discussions).
- **At least one Q&A category must exist** for answers to be accepted.
- **Tokens need `repo` and `write:discussion` scopes.**
- **Secondary accounts must have write access to the repository** (or be added as collaborators).
- **Galaxy Brain requires two accepted answers**; the script creates the exact number you specify.
- Achievements may take several hours to appear on your profile.

---

<div align="center">

**Copyright [Shinei Nouzen](https://github.com/Shineii86) All Rights Reserved.**  
*Made with ❤️ for GitHub achievement enthusiasts*


***Found this useful? Give it a Star! ⭐ [Shineii86/GalaxyBrain](https://github.com/Shineii86/GalaxyBrain)***
</div>